# Backtest and Risk Analysis (Phase 3)

This notebook evaluates the out-of-sample performance of the HMM regime-detection strategy against baselines, with a focus on fintech risk realism (VaR, CVaR, transaction costs, position sizing).

We strictly enforce:
1. **Walk-forward evaluation:** The model never sees future data.
2. **Signal lag:** Signals generated at close $t$ are executed at close $t+1$, earning the return of $t+2$.
3. **Realistic transaction costs:** We model bid-ask spread + commission instead of zero costs.
4. **Probability-weighted sizing:** Exposure is scaled by the predicted probability of favorable regimes, avoiding binary all-or-nothing trades.


In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.data.loader import load_config, download_prices
from src.features.engineering import build_features, FEATURE_COLUMNS
from src.models.hmm import walk_forward_predict
from src.models.labeling import smooth_regime_labels
from src.backtest.engine import run_backtest
from src.backtest.risk import compute_target_weight_proba
from src.backtest.baselines import get_buy_and_hold, get_200dma_filter
from src.backtest.metrics import get_all_metrics

cfg = load_config('../config/config.yaml')
data = download_prices(cfg)


## 1. Headline Results & Risk Metrics

We compute the performance of the strategy vs Buy & Hold and a 200-Day Moving Average (200DMA) filter.
Metrics include **Historical VaR & CVaR** (95% and 99%) as well as **Cornish-Fisher VaR & CVaR**.
The divergence between Historical and Cornish-Fisher VaR highlights the skewness and fat tails in strategy returns.

*Note: All risk figures are Daily. The standard $\sqrt{T}$ scaling rule is omitted as it often understates risk under volatility clustering.*


In [ ]:
results = []
ann_factor = 252

# Cache backtest results for later cycle breakdown
bt_results = {}

for asset_name in ['spx', 'nasdaq', 'gold', 'bitcoin']:
    if asset_name not in data: continue
    asset = data[asset_name]
    
    features = build_features(asset, asset['VIX_Close'])
    features['returns'] = asset['Close'].pct_change()
    features['Close'] = asset['Close']
    features = features.dropna()
    
    # 1. HMM Predictions
    raw_labels, out_probs, folds_info = walk_forward_predict(features, FEATURE_COLUMNS, cfg, asset_name=asset_name)
    
    # Probability weights: Size based on P(Bull) using trailing window smoothing
    smoothed_prob = out_probs['Bull'].rolling(window=cfg['model']['smoothing_window']).mean().fillna(0)
    target_weights_prob = compute_target_weight_proba(pd.DataFrame({'Bull': smoothed_prob}))
    
    hmm_res = run_backtest(features, target_weights_prob, asset_name=asset_name, use_dynamic_costs=True)
    
    # Baselines
    bh_res = get_buy_and_hold(features, asset_name=asset_name)
    dma_res = get_200dma_filter(features, asset_name=asset_name)
    
    bt_results[asset_name] = {'hmm': hmm_res, 'bh': bh_res, 'dma': dma_res}
    
    m_bh = get_all_metrics(bh_res['net_return'], bh_res['executed_weight'], ann_factor, asset_name.upper() + ' (B&H)')
    m_dma = get_all_metrics(dma_res['net_return'], dma_res['executed_weight'], ann_factor, asset_name.upper() + ' (200DMA)')
    m_hmm = get_all_metrics(hmm_res['net_return'], hmm_res['executed_weight'], ann_factor, asset_name.upper() + ' (HMM)')
    
    results.extend([m_bh, m_dma, m_hmm])

df_res = pd.DataFrame(results)
display(df_res.style.format(precision=4))


## 2. Per-Market-Cycle Breakdown

We evaluate performance across known historical market cycles to ensure the strategy's edge isn't an artifact of one lucky period.

*Note: Where sample sizes are extremely short (e.g., the 2020 COVID crash), Sharpe ratios are purely descriptive, not inferential. For BTC-USD, data gaps for earlier cycles will result in NaNs.*


In [ ]:
cycles = {
    "Dot-com (2000-03)": ("2000-01-01", "2003-12-31"),
    "Pre-GFC bull (2004-07)": ("2004-01-01", "2007-12-31"),
    "GFC (2008-09)": ("2008-01-01", "2009-12-31"),
    "2010s bull (2013-19)": ("2013-01-01", "2019-12-31"),
    "COVID crash (2020)": ("2020-01-01", "2020-12-31"),
    "2022 rate-hike bear": ("2022-01-01", "2022-12-31")
}

cycle_results = []

for c_name, (start, end) in cycles.items():
    for asset_name in ['spx', 'nasdaq', 'gold', 'bitcoin']:
        if asset_name not in bt_results: continue
        
        hmm_res = bt_results[asset_name]['hmm']
        mask = (hmm_res.index >= start) & (hmm_res.index <= end)
        
        # If less than 20 days of data in cycle, mark as gap
        if mask.sum() < 20: 
            cycle_results.append({
                "Cycle": c_name,
                "Asset": asset_name.upper(),
                "CAGR": np.nan, "Sharpe": np.nan, "Max DD": np.nan, 
                "Time in Market": np.nan, "Ann. Turnover": np.nan
            })
            continue
            
        sub_res = hmm_res[mask]
        metrics = get_all_metrics(sub_res['net_return'], sub_res['executed_weight'], ann_factor, asset_name.upper())
        
        cycle_results.append({
            "Cycle": c_name,
            "Asset": asset_name.upper(),
            "CAGR": metrics["CAGR"],
            "Sharpe": metrics["Sharpe"],
            "Max DD": metrics["Max DD"],
            "Time in Market": metrics["Time in Market"],
            "Ann. Turnover": metrics["Ann. Turnover"]
        })

df_cycle = pd.DataFrame(cycle_results)
display(df_cycle.set_index(['Cycle', 'Asset']).style.format(precision=4, na_rep="GAP"))


## 3. Regime Transition Analysis (2008 vs 2020)

How quickly does the model flag a crisis relative to the drawdown's start?
Trailing smoothing costs lag by construction — here we quantify that lag in days for the S&P 500 during the 2008 GFC and the 2020 COVID crash.


In [ ]:
asset_name = 'spx'
features = build_features(data[asset_name], data[asset_name]['VIX_Close'])
features['returns'] = data[asset_name]['Close'].pct_change()
features = features.dropna()

raw_labels, _, _ = walk_forward_predict(features, FEATURE_COLUMNS, cfg, asset_name=asset_name)
smooth_labels = smooth_regime_labels(raw_labels, window_size=cfg['model']['smoothing_window'])

events = {
    "2008 GFC": pd.to_datetime('2007-10-09'),  # S&P 500 peak before GFC
    "2020 COVID": pd.to_datetime('2020-02-19') # S&P 500 peak before COVID crash
}

for ev_name, peak_date in events.items():
    print(f"\n--- {ev_name} (Peak: {peak_date.date()}) ---")
    
    crisis_raw = raw_labels[(raw_labels.index >= peak_date) & (raw_labels == 'Crisis')]
    crisis_smooth = smooth_labels[(smooth_labels.index >= peak_date) & (smooth_labels == 'Crisis')]
    
    if not crisis_raw.empty:
        lag_raw = (crisis_raw.index[0] - peak_date).days
        print(f"Raw label flagged Crisis on {crisis_raw.index[0].date()} (Lag: {lag_raw} days)")
    
    if not crisis_smooth.empty:
        lag_smooth = (crisis_smooth.index[0] - peak_date).days
        print(f"Smoothed label flagged Crisis on {crisis_smooth.index[0].date()} (Lag: {lag_smooth} days)")
